# Complete the Substack archive workbook

This free notebook applies the committed October subtitle patch to the audited Excel master. It does not change your GitHub repository. At the end it downloads a completed Excel workbook and a blank-subtitle audit for you to upload to GitHub.

## What to do
1. Download `1070_posts_with_war_day_fixed.xlsx` and `october_subtitle_patch_2026.csv` from the repository.
2. Run this notebook in Google Colab: **Runtime → Run all**.
3. When the upload box appears, select both downloaded files.
4. Colab will check the master file, apply the patch, and download the finished workbook plus audit.
5. Upload the finished workbook to the repository, replacing the old `1070_posts_with_war_day_fixed.xlsx`. Upload the audit too if you like.

The notebook stops rather than guessing if the workbook is not the expected master or a row does not match uniquely.

In [ ]:
!pip -q install openpyxl

from google.colab import files
from pathlib import Path
from openpyxl import load_workbook
from copy import copy
from datetime import datetime, date
import csv, hashlib, io

EXPECTED_GIT_BLOB = '1a7a38e3e62571151985f8e2294748faf347bd26'
MASTER_NAME = '1070_posts_with_war_day_fixed.xlsx'
PATCH_NAME = 'october_subtitle_patch_2026.csv'

print('Select BOTH files: the Excel master and the October patch CSV.')
uploaded = files.upload()
if MASTER_NAME not in uploaded or PATCH_NAME not in uploaded:
    raise RuntimeError('Both exact filenames are required. No workbook was changed.')

master_bytes = uploaded[MASTER_NAME]
git_blob = hashlib.sha1(b'blob ' + str(len(master_bytes)).encode() + b'\\0' + master_bytes).hexdigest()
if git_blob != EXPECTED_GIT_BLOB:
    raise RuntimeError('This is not the audited master workbook version. Stop here; do not upload a replacement.')

patch_rows = list(csv.DictReader(io.StringIO(uploaded[PATCH_NAME].decode('utf-8-sig')), delimiter=';'))
if not patch_rows or not {'date','day_of_war','title','subtitle','url','action'}.issubset(patch_rows[0]):
    raise RuntimeError('Patch file headers are not as expected.')

wb = load_workbook(io.BytesIO(master_bytes))
aliases = {
    'date': {'date'},
    'day_of_war': {'day_of_war','day','war_day'},
    'title': {'title','post_title'},
    'subtitle': {'subtitle','sub_title'},
    'url': {'url','canonical_url','link'}
}
ws = cols = header_row = None
for sheet in wb.worksheets:
    for rr in range(1, min(sheet.max_row, 20)+1):
        found = {}
        for cc in range(1, sheet.max_column+1):
            label = str(sheet.cell(rr,cc).value or '').strip().lower().replace(' ','_').replace('-','_')
            for field, names in aliases.items():
                if label in names: found[field] = cc
        if len(found) == 5:
            ws, cols, header_row = sheet, found, rr
            break
    if ws is not None: break
if ws is None: raise RuntimeError('Could not identify date/day/title/subtitle/URL columns. No output produced.')

def dkey(v):
    if isinstance(v, (datetime,date)): return v.strftime('%Y-%m-%d')
    return str(v or '').strip()[:10]
def wkey(v):
    try: return str(int(v))
    except (TypeError,ValueError): return str(v or '').strip()
def rowkey(rr):
    return (dkey(ws.cell(rr,cols['date']).value), wkey(ws.cell(rr,cols['day_of_war']).value), str(ws.cell(rr,cols['title']).value or '').strip(), str(ws.cell(rr,cols['url']).value or '').strip())

filled = added = intentional = 0
for item in patch_rows:
    key = (item['date'], item['day_of_war'], item['title'].strip(), item['url'].strip())
    hits = [rr for rr in range(header_row+1, ws.max_row+1) if rowkey(rr) == key]
    if item['action'] == 'add_verified_row':
        if hits:
            if len(hits) != 1 or str(ws.cell(hits[0],cols['subtitle']).value or '').strip() != item['subtitle']:
                raise RuntimeError('Day 1709 conflict. No output produced.')
            continue
        if any(wkey(ws.cell(rr,cols['day_of_war']).value) == '1709' for rr in range(header_row+1,ws.max_row+1)):
            raise RuntimeError('A different Day 1709 row already exists. No output produced.')
        rr = ws.max_row + 1
        for cc in range(1,ws.max_column+1):
            if ws.cell(rr-1,cc).has_style: ws.cell(rr,cc)._style = copy(ws.cell(rr-1,cc)._style)
        for field in ('date','day_of_war','title','subtitle','url'):
            value = item[field]
            if field == 'date': value = datetime.strptime(value,'%Y-%m-%d').date()
            if field == 'day_of_war': value = int(value)
            ws.cell(rr,cols[field]).value = value
        added += 1
        continue
    if len(hits) != 1:
        raise RuntimeError(f'Expected exactly one matching row, found {len(hits)} for {key}. No output produced.')
    cell = ws.cell(hits[0],cols['subtitle'])
    existing = str(cell.value or '').strip()
    if item['action'] == 'intentional_blank':
        if existing: raise RuntimeError('Day 1696 was expected to be blank but is not. No output produced.')
        intentional += 1
    elif item['action'] == 'update':
        if existing and existing != item['subtitle']:
            raise RuntimeError(f'Refusing to overwrite a different subtitle for {key}. No output produced.')
        if not existing:
            cell.value = item['subtitle']
            filled += 1
    else:
        raise RuntimeError('Unknown patch action: ' + item['action'])

blank_rows = []
data_rows = 0
for rr in range(header_row+1,ws.max_row+1):
    k = rowkey(rr)
    if any(k):
        data_rows += 1
        if not str(ws.cell(rr,cols['subtitle']).value or '').strip(): blank_rows.append((rr,k))

out_name = '1070_posts_with_war_day_fixed_completed.xlsx'
wb.save(out_name)
audit_name = 'SUBTITLE_AUDIT_2026-10-30.md'
with open(audit_name,'w',encoding='utf-8') as f:
    f.write('# Substack archive subtitle audit — 2026-10-30\\n\\n')
    f.write(f'- Worksheet: {ws.title}\\n- Data rows: {data_rows}\\n- Newly filled subtitles: {filled}\\n- Day 1709 rows added: {added}\\n- Intentional blank confirmed (Day 1696): {intentional}\\n- Remaining blank subtitle cells: {len(blank_rows)}\\n\\n')
    f.write('## Remaining blanks\\n\\n')
    for rr,k in blank_rows: f.write(f'- Row {rr}: {k[0]} | Day {k[1]} | {k[2]} | {k[3]}\\n')

print('CHECK COMPLETE')
print(f'Worksheet: {ws.title} | Data rows: {data_rows} | Subtitles filled: {filled} | Day 1709 added: {added}')
print(f'Day 1696 intentional blank: {intentional} | Remaining blank subtitles: {len(blank_rows)}')
files.download(out_name)
files.download(audit_name)
